In [ ]:
import pandas as pd
import numpy as np
import os

data_path = "../data/"
shopping_file = "customer_shopping_data.csv"
behavior_file = "ecommerce_customer_behavior.csv"
reviews_file = "womens_clothing_reviews.csv"
shopping = pd.read_csv(os.path.join(data_path, shopping_file))
behavior = pd.read_csv(os.path.join(data_path, behavior_file))
reviews = pd.read_csv(os.path.join(data_path, reviews_file))

print(" Tebrikler! Tüm veri setleri başarıyla yüklendi.")
print(f"Shopping: {shopping.shape}")
print(f"Behavior: {behavior.shape}")
print(f"Reviews: {reviews.shape}")

In [ ]:

def clean_columns(df):
    df.columns = (
        df.columns
        .str.lower()         
        .str.strip()         
        .str.replace(" ", "_") 
        .str.replace("-", "_") 
    )
    return df

shopping = clean_columns(shopping)
behavior = clean_columns(behavior)
reviews = clean_columns(reviews)

print(" Kolon isimleri standart hale getirildi!")
print("Shopping Kolonları:", shopping.columns.tolist())
print("\nBehavior Kolonları:", behavior.columns.tolist())
print("\nReviews Kolonları:", reviews.columns.tolist())

In [ ]:

for name, df in {"Shopping": shopping, "Behavior": behavior, "Reviews": reviews}.items():
    print(f"--- {name} Eksik Veri Sayısı ---")
    print(df.isnull().sum())
    print("\n")

In [ ]:

def create_age_groups(df):
    bins = [0, 18, 25, 35, 45, 60, 100]
    labels = ["0-18", "19-25", "26-35", "36-45", "46-60", "60+"]
    df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)
    return df

shopping = create_age_groups(shopping)
behavior = create_age_groups(behavior)
reviews = create_age_groups(reviews)

shopping['total_amount'] = shopping['price'] * shopping['quantity']
shopping['spending_segment'] = pd.qcut(shopping['total_amount'], q=3, labels=["Low", "Medium", "High"])

shopping = shopping.rename(columns={'category': 'main_category'})
behavior = behavior.rename(columns={'product_category': 'main_category'}) 
reviews = reviews.rename(columns={'class_name': 'main_category'})

print(" Özellikler üretildi ve kategoriler isimlendirildi!")

In [ ]:

print("Behavior kolonları:", behavior.columns.tolist())
if 'product_category' in behavior.columns:
    behavior = behavior.rename(columns={'product_category': 'main_category'})
elif 'category' in behavior.columns:
    behavior = behavior.rename(columns={'category': 'main_category'})
if 'class_name' in reviews.columns:
    reviews = reviews.rename(columns={'class_name': 'main_category'})

print(" Kolon isimleri tekrar kontrol edildi ve güncellendi.")

In [ ]:

shopping_summary = shopping.groupby(['age_group', 'gender', 'main_category']).agg({
    'total_amount': 'mean',
    'quantity': 'sum'
}).reset_index().rename(columns={'total_amount': 'avg_shopping_spend'})
behavior_summary = behavior.groupby(['age_group', 'gender']).agg({
    'average_rating': 'mean'
}).reset_index().rename(columns={'average_rating': 'behavior_satisfaction'})
reviews_summary = reviews.groupby(['age_group', 'main_category']).agg({
    'rating': 'mean'
}).reset_index().rename(columns={'rating': 'review_rating'})

print(" Özet tablolar başarıyla oluşturuldu!")
display(behavior_summary.head())

In [ ]:

merged_df = pd.merge(shopping_summary, behavior_summary, on=['age_group', 'gender'], how='inner')
final_df = pd.merge(merged_df, reviews_summary, on=['age_group', 'main_category'], how='left')
final_df['review_rating'] = final_df['review_rating'].fillna(final_df['review_rating'].mean())
print(" MUHTEŞEM! 3 Dataset artık tek bir tabloda.")
print(f"Final Tablo Boyutu: {final_df.shape}")
display(final_df.head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
sns.barplot(data=final_df, x='age_group', y='avg_shopping_spend', hue='gender')
plt.title('Yaş Grubu ve Cinsiyete Göre Ortalama Harcama')
plt.xticks(rotation=45)
plt.subplot(1, 2, 2)
sns.scatterplot(data=final_df, x='review_rating', y='avg_shopping_spend', hue='age_group', s=100)
plt.title('Yorum Puanı vs. Harcama İlişkisi')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

median_spend = final_df['avg_shopping_spend'].median()
final_df['spend_level'] = final_df['avg_shopping_spend'].apply(lambda x: 1 if x > median_spend else 0)

le = LabelEncoder()
model_data = final_df.copy()
model_data['gender'] = le.fit_transform(model_data['gender'])
model_data['main_category'] = le.fit_transform(model_data['main_category'])
model_data['age_group'] = le.fit_transform(model_data['age_group'].astype(str))

X = model_data[['age_group', 'gender', 'main_category', 'behavior_satisfaction', 'review_rating']]
y = model_data['spend_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print(" Model eğitildi! Şimdi sonuçlara bakalım.")

In [ ]:

y_pred = model.predict(X_test)

print("--- MODEL BAŞARI RAPORU ---")
print(classification_report(y_test, y_pred))

importance = pd.DataFrame({
    'Özellik': X.columns,
    'Önem Düzeyi': model.feature_importances_
}).sort_values(by='Önem Düzeyi', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=importance, x='Önem Düzeyi', y='Özellik')
plt.title('Harcama Seviyesini Etkileyen Faktörler')
plt.show()

In [ ]:

df_final_pro = shopping.copy()
behavior_map = behavior.groupby(['age_group', 'gender'])['average_rating'].mean().to_dict()
df_final_pro['predicted_satisfaction'] = df_final_pro.set_index(['age_group', 'gender']).index.map(behavior_map)
reviews_map = reviews.groupby(['age_group', 'main_category'])['rating'].mean().to_dict()
df_final_pro['category_rating'] = df_final_pro.set_index(['age_group', 'main_category']).index.map(reviews_map)
df_final_pro['predicted_satisfaction'] = df_final_pro['predicted_satisfaction'].fillna(df_final_pro['predicted_satisfaction'].mean())
df_final_pro['category_rating'] = df_final_pro['category_rating'].fillna(df_final_pro['category_rating'].mean())

print(f" Profesyonel veri seti hazır! Satır sayısı: {len(df_final_pro)}")
display(df_final_pro.head())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
df_final_pro['spending_segment'] = pd.qcut(df_final_pro['total_amount'], q=3, labels=[0, 1, 2])
X = pd.get_dummies(df_final_pro[['age_group', 'gender', 'main_category', 'predicted_satisfaction', 'category_rating']])
y = df_final_pro['spending_segment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_pro = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model_pro.fit(X_train, y_train)
print(" 99,000 satırlık profesyonel model eğitildi!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df_pro = shopping.copy()
behavior_map = behavior.groupby(['age_group', 'gender'])['average_rating'].mean().to_dict()
reviews_map = reviews.groupby(['age_group', 'main_category'])['rating'].mean().to_dict()
df_pro['predicted_satisfaction'] = df_pro.set_index(['age_group', 'gender']).index.map(behavior_map)
df_pro['category_rating'] = df_pro.set_index(['age_group', 'main_category']).index.map(reviews_map)
df_pro['predicted_satisfaction'].fillna(df_pro['predicted_satisfaction'].mean(), inplace=True)
df_pro['category_rating'].fillna(df_pro['category_rating'].mean(), inplace=True)
print(f"--- ANALİZ ÖZETİ ({len(df_pro)} Satır) ---")
print(df_pro[['total_amount', 'predicted_satisfaction', 'category_rating']].describe())
plt.figure(figsize=(12, 6))
sns.kdeplot(data=df_pro, x='total_amount', hue='age_group', fill=True)
plt.title('Yaş Gruplarına Göre Harcama Yoğunluğu (Büyük Veri Analizi)')
plt.xlabel('Toplam Harcama (TL)')
plt.ylabel('Yoğunluk')
plt.show()

In [ ]:
import pandas as pd
print("--- MEVCUT KOLONLARINIZ ---")
print(df.columns.tolist())
print("---------------------------")
cat_candidates = [c for c in df.columns if any(x in c.lower() for x in ['cat', 'class', 'type'])]

if not cat_candidates:
    cat_candidates = df.select_dtypes(include=['object']).columns.tolist()
if cat_candidates:
    final_cat_col = cat_candidates[0]
    print(f" Bulunan en mantıklı kategori kolonu: '{final_cat_col}'")
    if 'age_group' not in df.columns:
        bins = [0, 18, 25, 35, 45, 60, 100]
        labels = ["0-18", "19-25", "26-35", "36-45", "46-60", "60+"]
        df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)
    age_cat_summary = df.groupby(['age_group', final_cat_col], observed=True).size().reset_index(name='islem_sayisi')
    age_cat_summary = age_cat_summary.rename(columns={final_cat_col: 'category'})
    display(age_cat_summary.sort_values(by=['age_group', 'islem_sayisi'], ascending=[True, False]).head(10))
else:
    print(" Maalesef hiçbir kategori benzeri kolon bulunamadı. Lütfen kolon listesini kontrol et!")

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
@st.cache_data
def load_data():
    df = pd.read_csv("data/customer_shopping_data.csv")
    df.columns = df.columns.str.strip().str.lower()
    df['total_amount'] = df['price'] * df['quantity']
    bins = [0, 18, 25, 35, 45, 60, 100]
    labels = ["0-18", "19-25", "26-35", "36-45", "46-60", "60+"]
    df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)
    
    return df

try:
    df = load_data()

    st.title(" Yaş Grupları Arası Kategori Düellosu")
    st.markdown("---") 
    col_a, col_b = st.columns(2)
    with col_a:
        yas_1 = st.selectbox("Birinci Yaş Grubu Seçin", df['age_group'].unique(), index=0)
        df_1 = df[df['age_group'] == yas_1]
        kat_1 = df_1['category'].value_counts().reset_index()
        kat_1.columns = ['kategori', 'adet']
        st.write(f" **{yas_1} Yaş** Tercihleri")
        fig1 = px.bar(kat_1, x='kategori', y='adet', color='kategori', text_auto=True)
        st.plotly_chart(fig1, use_container_width=True)

    with col_b:
        yas_2 = st.selectbox("İkinci Yaş Grubu Seçin", df['age_group'].unique(), index=len(df['age_group'].unique())-1)
        df_2 = df[df['age_group'] == yas_2]
        kat_2 = df_2['category'].value_counts().reset_index()
        kat_2.columns = ['kategori', 'adet']
        
        st.write(f" **{yas_2} Yaş** Tercihleri")
        fig2 = px.bar(kat_2, x='kategori', y='adet', color='kategori', text_auto=True)
        st.plotly_chart(fig2, use_container_width=True)

except Exception as e:
    st.error(f" Bir hata oluştu: {e}")